# Syngenta India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** api.smartrecruiters.com/SyngentaGroup

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-02 10:34:46
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Syngenta"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Syngenta/Outputs/2026_04_02


In [4]:
from bs4 import BeautifulSoup

TARGET_COLS = [
    "job_id", "title", "location_city", "State", "Country",
    "Job description", "Accountabilities",
    "Qualifications Required", "Qualification Desired",
    "business_unit", "job_url", "source_api_url",
    "skills_required", "skills_preferred",
    "min_years_experience", "seniority_level", "work_mode",
    "employment_type", "degree_required", "degree_preferred_field",
    "salary_min", "salary_max", "date_posted", "is_active", "industry",
]

def parse_jd_sections(jd_html):
    """Split jobDescription HTML into intro and accountabilities by heading."""
    if not jd_html:
        return "", ""
    soup = BeautifulSoup(jd_html, "html.parser")
    intro_parts, acc_parts = [], []
    in_acc = False
    for tag in soup.children:
        if not hasattr(tag, "get_text"):
            continue
        text = tag.get_text(" ", strip=True)
        strong = tag.find("strong") if hasattr(tag, "find") else None
        if strong and "accountabilit" in strong.get_text().lower():
            in_acc = True
            continue
        if in_acc:
            acc_parts.append(text)
        else:
            if text:
                intro_parts.append(text)
    return " ".join(intro_parts).strip(), " ".join(acc_parts).strip()

def parse_qual_sections(qual_html):
    """Split qualifications HTML into Required and Desired by heading."""
    if not qual_html:
        return "", ""
    soup = BeautifulSoup(qual_html, "html.parser")
    req_parts, des_parts = [], []
    mode = None
    for tag in soup.children:
        if not hasattr(tag, "get_text"):
            continue
        text = tag.get_text(" ", strip=True)
        strong = tag.find("strong") if hasattr(tag, "find") else None
        if strong:
            label = strong.get_text().lower()
            if "required" in label:
                mode = "required"
                continue
            elif "desired" in label or "preferred" in label:
                mode = "desired"
                continue
        if mode == "required" and text:
            req_parts.append(text)
        elif mode == "desired" and text:
            des_parts.append(text)
    return " ".join(req_parts).strip(), " ".join(des_parts).strip()

def scrape_syngenta_detailed(company_id="SyngentaGroup", max_jobs=500, country=""):
    import requests, time, random

    base_api = f"https://api.smartrecruiters.com/v1/companies/{company_id}/postings"
    session = requests.Session()
    session.headers.update({
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
    })

    jobs = []
    offset = 0
    limit = 100

    print(f"Scraping Syngenta via SmartRecruiters API (country='{country}' — {'broad/global' if not country else country})")

    while offset < max_jobs:
        params = {"limit": limit, "offset": offset}
        if country:
            params["country"] = country

        try:
            resp = session.get(base_api, params=params, timeout=30)
            if resp.status_code != 200:
                print(f"  [ERROR] HTTP {resp.status_code}")
                break

            data = resp.json()
            content = data.get("content", [])
            if not content:
                break

            print(f"  Page offset={offset}: {len(content)} jobs")

            for posting in content:
                loc          = posting.get("location", {})
                city         = loc.get("city", "") or ""
                state        = loc.get("region", "") or ""
                iso2         = (loc.get("country", "") or "").lower()
                country_full = ISO2_TO_COUNTRY.get(iso2, iso2.upper()) if iso2 else ""
                posting_id   = posting.get("id", "")

                job_desc = acc = qual_req = qual_des = ""

                if posting_id:
                    try:
                        det = session.get(f"{base_api}/{posting_id}", timeout=20)
                        if det.status_code == 200:
                            sections  = det.json().get("jobAd", {}).get("sections", {})
                            jd_html   = sections.get("jobDescription", {}).get("text", "")
                            qual_html = sections.get("qualifications",  {}).get("text", "")
                            job_desc, acc      = parse_jd_sections(jd_html)
                            qual_req, qual_des = parse_qual_sections(qual_html)
                        time.sleep(random.uniform(0.3, 0.8))
                    except Exception as e:
                        print(f"    [WARN] {posting_id}: {e}")

                # Combined JD text for inference
                full_jd = " ".join(filter(None, [job_desc, acc, qual_req, qual_des]))
                title   = posting.get("name", "")

                # Inference (uses scraper_utils functions loaded at import time)
                skills_all = extract_skills(full_jd)
                half       = max(1, len(skills_all) // 2)
                min_exp, _ = infer_exp(full_jd)

                # Posted date
                created     = posting.get("releasedDate", posting.get("createdOn", ""))
                posted_date = created[:10] if created else datetime.now().strftime("%Y-%m-%d")

                company_slug  = posting.get("company", {}).get("identifier", company_id)
                job_url       = f"https://jobs.smartrecruiters.com/{company_slug}/{posting_id}"
                dept          = posting.get("department", {})
                business_unit = dept.get("label", dept.get("id", "")) if isinstance(dept, dict) else str(dept)

                jobs.append({
                    "job_id":                  str(posting.get("refNumber", posting_id)),
                    "title":                   title,
                    "location_city":           city,
                    "State":                   state,
                    "Country":                 country_full,
                    "Job description":         job_desc,
                    "Accountabilities":        acc,
                    "Qualifications Required": qual_req,
                    "Qualification Desired":   qual_des,
                    "business_unit":           business_unit,
                    "job_url":                 job_url,
                    "source_api_url":          base_api,
                    "skills_required":         " | ".join(skills_all[:half]),
                    "skills_preferred":        " | ".join(skills_all[half:]),
                    "min_years_experience":    min_exp,
                    "seniority_level":         infer_seniority(title, full_jd),
                    "work_mode":               infer_work_mode(full_jd + " " + city),
                    "employment_type":         infer_emp_type(full_jd + " " + title),
                    "degree_required":         infer_degree(full_jd),
                    "degree_preferred_field":  infer_degree_field(full_jd),
                    "salary_min":              "",
                    "salary_max":              "",
                    "date_posted":             posted_date,
                    "is_active":               True,
                    "industry":                "Agriculture / Agrochemical",
                })

            if len(content) < limit:
                break
            offset += limit
            time.sleep(random.uniform(0.5, 1.0))

        except Exception as e:
            print(f"  [ERROR] {e}")
            break

    print(f"  Total Syngenta jobs scraped: {len(jobs)}")
    return jobs

print("=" * 60)
print("SYNGENTA JOB SCRAPER — Detailed Section Extraction")
print("ATS: SmartRecruiters (api.smartrecruiters.com)")
print("=" * 60)

syngenta_jobs = scrape_syngenta_detailed(
    company_id="SyngentaGroup",
    max_jobs=500,
    country=COUNTRY_CODE,
)


SYNGENTA JOB SCRAPER — Detailed Section Extraction
ATS: SmartRecruiters (api.smartrecruiters.com)
Scraping Syngenta via SmartRecruiters API (country='' — broad/global)


  Page offset=0: 100 jobs


  Page offset=100: 100 jobs


  Page offset=200: 100 jobs


  Page offset=300: 100 jobs


  Page offset=400: 28 jobs


  Total Syngenta jobs scraped: 428


In [5]:
import pandas as pd
from pathlib import Path
from datetime import datetime

if syngenta_jobs:
    df_syngenta = pd.DataFrame(syngenta_jobs, columns=TARGET_COLS)
    df_syngenta = df_syngenta.drop_duplicates(subset=["job_id", "title", "job_url"], keep="first")

    date_tag  = datetime.now().strftime("%Y-%m-%d")
    out_dir   = Path(OUTPUT_DIR)
    csv_path  = out_dir / f"Syngenta_jobs_{date_tag}.csv"
    xlsx_path = out_dir / f"Syngenta_jobs_{date_tag}.xlsx"

    df_syngenta.to_csv(csv_path,  index=False, encoding="utf-8")
    df_syngenta.to_excel(xlsx_path, index=False, engine="openpyxl")

    def has_value(x):
        return x not in [None, "", float("nan")] and str(x) not in ("nan", "None")

    print(f"[OK] Saved {len(df_syngenta)} jobs -> {csv_path.name}")
    print(f"     Total columns: {len(df_syngenta.columns)}")
    print(f"     Has State:                {df_syngenta['State'].apply(bool).sum()}/{len(df_syngenta)}")
    print(f"     Has Accountabilities:     {df_syngenta['Accountabilities'].apply(lambda x: bool(x) and len(str(x))>20).sum()}/{len(df_syngenta)}")
    print(f"     Has Qual Required:        {df_syngenta['Qualifications Required'].apply(lambda x: bool(x) and len(str(x))>20).sum()}/{len(df_syngenta)}")
    print(f"     Has Qual Desired:         {df_syngenta['Qualification Desired'].apply(lambda x: bool(x) and len(str(x))>20).sum()}/{len(df_syngenta)}")
    print(f"     Has skills_required:      {df_syngenta['skills_required'].apply(bool).sum()}/{len(df_syngenta)}")
    print(f"     Has seniority_level:      {df_syngenta['seniority_level'].apply(bool).sum()}/{len(df_syngenta)}")
    print(f"     Has min_years_experience: {df_syngenta['min_years_experience'].apply(has_value).sum()}/{len(df_syngenta)}")

    print(f"\nSample (first 3 rows):")
    preview = ["title", "location_city", "State", "Country", "seniority_level", "work_mode", "business_unit"]
    print(df_syngenta[preview].head(3).to_string())
else:
    print("[WARN] No jobs found.")
    df_syngenta = None


[OK] Saved 428 jobs -> Syngenta_jobs_2026-04-02.csv
     Total columns: 25
     Has State:                399/428
     Has Accountabilities:     132/428
     Has Qual Required:        60/428
     Has Qual Desired:         49/428
     Has skills_required:      154/428
     Has seniority_level:      428/428
     Has min_years_experience: 60/428

Sample (first 3 rows):
                                  title location_city     State        Country seniority_level work_mode business_unit
0                 Biological Specialist         Malta  ILLINOIS  United States          junior    onsite    Commercial
1      Senior Dietary Residue Scientist    Greensboro        NC  United States            lead    remote   Development
2  Crop Protection Sales Representative         Colby        KS  United States            lead    onsite    Commercial
